# 10.04 · Evaluating RAG

**Chapter:** Evaluation · **Milestone:** M5

## Goals
- [ ] Retrieval metrics: recall@k, precision@k, MRR, nDCG — implemented, not imported
- [ ] Generation metrics: faithfulness (claims supported by sources), answer correctness, answer relevance
- [ ] Evaluate retrieval and generation **separately**, then end to end
- [ ] Know the libraries: Ragas, DeepEval, TruLens, Phoenix evals

## Resources
- [Pinecone – Evaluation measures in IR](https://www.pinecone.io/learn/offline-evaluation/) — free
- [Ragas metrics docs](https://docs.ragas.io/en/stable/concepts/metrics/) · [DeepEval metrics](https://deepeval.com/docs/metrics-introduction) — free
- [RAGAS paper](https://arxiv.org/abs/2309.15217) — free

The RAG triad: **context relevance** (did we retrieve the right stuff?), **faithfulness / groundedness** (is the answer supported by what we
retrieved?), **answer relevance / correctness** (does it answer the question, correctly?).

In [ ]:
import json

import numpy as np
from pydantic import BaseModel, Field

from aieng import rag
from aieng.llm import client

c, MODEL = client()
EXTRA = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}
evalset = rag.eval_set()

## 1. Retrieval metrics
For a ranked list of retrieved ids and a set of relevant ids:
- **recall@k**: fraction of relevant docs found in the top k (or "hit rate": any relevant found)
- **precision@k**: fraction of the top k that is relevant
- **MRR**: 1 / rank of the first relevant result (rewards putting it first)
- **nDCG@k**: discounted gain by position, normalised by the ideal ranking

In [ ]:
def hit_at_k(ranked, relevant, k):
    return float(bool(set(ranked[:k]) & relevant))


def recall_at_k(ranked, relevant, k):
    return len(set(ranked[:k]) & relevant) / len(relevant)


def precision_at_k(ranked, relevant, k):
    return len(set(ranked[:k]) & relevant) / k


def mrr(ranked, relevant):
    return next((1 / (i + 1) for i, d in enumerate(ranked) if d in relevant), 0.0)


def ndcg_at_k(ranked, relevant, k):
    dcg = sum(1 / np.log2(i + 2) for i, d in enumerate(ranked[:k]) if d in relevant)
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg


assert mrr([5, 3, 9], {3}) == 0.5 and hit_at_k([5, 3, 9], {9}, 2) == 0.0 and ndcg_at_k([3, 5], {3}, 2) == 1.0


def retrieval_report(search_fn, k=5):
    m = {"hit@k": [], "recall@k": [], "precision@k": [], "MRR": [], "nDCG@k": []}
    for r in evalset:
        ranked = search_fn(r["question"], 20)
        m["hit@k"].append(hit_at_k(ranked, r["gold"], k)); m["recall@k"].append(recall_at_k(ranked, r["gold"], k))
        m["precision@k"].append(precision_at_k(ranked, r["gold"], k)); m["MRR"].append(mrr(ranked, r["gold"])); m["nDCG@k"].append(ndcg_at_k(ranked, r["gold"], k))
    return {name: round(float(np.mean(v)), 3) for name, v in m.items()}


print("dense bge-small:", retrieval_report(lambda q, n: [p for p, _, _ in rag.search(q, n)]))

Recall@k is low here because our proxy gold sets contain *every* passage mentioning the answer string (often 10+); hit@k is the more meaningful
number for this dataset. With exact labels (10.02's synthetic set: one gold passage) recall@k and hit@k coincide.

## 2. Faithfulness: is every claim supported by the sources?
Ragas-style: (1) split the answer into atomic claims, (2) check each claim against the retrieved context, (3) score = supported / total.
Catches hallucinations even when the answer happens to be correct (the model "knew" it, but didn't get it from the sources).

In [ ]:
class Claims(BaseModel):
    claims: list[str] = Field(description="atomic factual statements made in the answer")


class Support(BaseModel):
    reasoning: str
    supported: bool


def parse(prompt, schema):
    return c.chat.completions.parse(model=MODEL, temperature=0, max_tokens=600, response_format=schema, **EXTRA,
                                    messages=[{"role": "user", "content": prompt}]).choices[0].message.parsed


def faithfulness(answer, contexts):
    claims = parse(f"List the atomic factual claims in this answer (ignore citations like [12]).\n\nAnswer: {answer}", Claims).claims
    if not claims:
        return 1.0, []
    ctx = "\n\n".join(contexts)
    verdicts = [(cl, parse(f"Context:\n{ctx}\n\nIs this claim directly supported by the context?\nClaim: {cl}", Support).supported) for cl in claims]
    return sum(v for _, v in verdicts) / len(verdicts), verdicts


answer, hits = rag.answer("When was Abraham Lincoln born and where?", max_tokens=300)
score, verdicts = faithfulness(answer, [t for _, _, t in hits])
print(answer, "\n")
for claim, ok in verdicts:
    print(("✓" if ok else "✗"), claim)
print("faithfulness:", round(score, 2))

In [ ]:
# A deliberately unfaithful answer: correct-sounding facts that are NOT in the retrieved context
fake = "Lincoln was born on February 12, 1809, in Kentucky. He later became a skilled violinist and toured Europe in 1830."
print("faithfulness of a partly invented answer:", round(faithfulness(fake, [t for _, _, t in hits])[0], 2))

## 3. End-to-end: correctness + faithfulness on a sample
Combine: code check (citations valid), LLM judge for correctness vs reference (10.03), faithfulness. Small sample — each answer costs several judge calls.

In [ ]:
class Verdict(BaseModel):
    critique: str
    correct: bool


def correct(question, reference, answer):
    return parse(f"Question: {question}\nReference: {reference}\nAnswer: {answer}\n"
                 "Is the answer factually consistent with the reference (paraphrases are fine)?", Verdict).correct


results = []
for r in evalset[:15]:
    ans, hits = rag.answer(r["question"], max_tokens=300)
    f, _ = faithfulness(ans, [t for _, _, t in hits])
    results.append({"retrieved_gold": bool({p for p, _, _ in hits} & r["gold"]), "correct": correct(r["question"], r["answer"], ans), "faithfulness": f})

summary = {k: round(float(np.mean([x[k] for x in results])), 2) for k in results[0]}
print(summary)
both = [x for x in results if x["retrieved_gold"]]
print(f"correct when gold retrieved: {np.mean([x['correct'] for x in both]):.0%} of {len(both)} | "
      f"when not: {np.mean([x['correct'] for x in results if not x['retrieved_gold']] or [0]):.0%}")

The conditional numbers tell you where to invest: if answers are good when retrieval works, fix retrieval; if they're bad even with the gold
passage in context, fix generation (prompt, model, context packing).

## 4. Libraries
Same metrics, packaged with prompts that have been tuned on many datasets. They're LLM-judged, so **calibrate them too** (10.03).
Ragas and DeepEval conflict with this repo's pinned dependencies, so run them in an isolated environment:
```bash
uv run --isolated --with ragas --with langchain-openai python my_ragas_eval.py
uv run --isolated --with deepeval python my_deepeval_eval.py
```
```python
# DeepEval (pytest-style)
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
case = LLMTestCase(input=q, actual_output=answer, retrieval_context=contexts)
FaithfulnessMetric(threshold=0.8).measure(case)

# Ragas
from ragas import evaluate, EvaluationDataset
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextRecall
evaluate(EvaluationDataset.from_list(rows), metrics=[Faithfulness(), LLMContextRecall()], llm=..., embeddings=...)
```
Also: **TruLens** (RAG triad), **Arize Phoenix evals** (11.01), **Inspect AI** (10.05).

## Exercises
1. Compute retrieval metrics on 10.02's synthetic set (`data/eval-synth-v1-dev.jsonl`, exact gold) for dense, BM25, hybrid and hybrid+rerank (07).
2. Run faithfulness with a hosted judge over 50 answers. Read every claim marked unsupported: real hallucination, or judge error?
3. Implement **answer relevance**: generate 3 questions from the answer, embed them, and average their cosine similarity with the original question.

In [ ]:
# your code here